In [1]:
import pandas as pd
import random

In [133]:
# import data from conditions file csv

conditions = pd.read_excel('/Users/camillaucheomaenwereuzor/Desktop/Experiments/proj1.5_onesacc/short_trial_variables_right.xlsx') # change last word to 'right' or 'left' depending on which file you want to generate
                                                                                                                             # change final csv filename at the end accordingly before running
# conditions.head(16)

In [134]:
### setup core variables
n_conditions = 8   # n rows in conditions file

n_blocks = 8
trials_per_block = 120
tot_trials = trials_per_block * n_blocks
extra_trials_per_block = 10

interrupt_per_block = 12
tot_interruptions = interrupt_per_block * n_blocks
interrupt_freq = tot_trials / tot_interruptions

avg_gap = int(trials_per_block / interrupt_per_block)   # average n trials between evenly-spaced interruptions

gap_range = list(range((avg_gap - 2), (avg_gap + 2) + 1))   # add ±2 jitter to gaps between interruptions - NB: the n of items in gap_range must be a factor of interrupt_per_block
if interrupt_per_block % len(gap_range) == 0:
    gap_range = gap_range
elif interrupt_per_block % 3 == 0:
    gap_range = [(avg_gap - 2), avg_gap, (avg_gap + 2)]
else:
    gap_range = list(range((avg_gap - 2), (avg_gap + 1) + 1))

print(f'''
    n conditions =  {n_conditions} 
    tot trials = {tot_trials} ; plus extra trials = {tot_trials + (extra_trials_per_block * n_blocks)}
    interrupt frequency = {interrupt_freq}% 
    n blocks = {n_blocks} 
    trials per block = {trials_per_block} ; plus extra trials = {trials_per_block + extra_trials_per_block}
    tot interruptions = {tot_interruptions} 
    interruptions per block = {interrupt_per_block} 
    average gap = {avg_gap} 
    gap range = {gap_range}
''')


    n conditions =  8 
    tot trials = 960 ; plus extra trials = 1040
    interrupt frequency = 10.0% 
    n blocks = 8 
    trials per block = 120 ; plus extra trials = 130
    tot interruptions = 96 
    interruptions per block = 12 
    average gap = 10 
    gap range = [8, 10, 12]



In [135]:
## shuffle trials in chunks of 8 and append to new dataframe to make entire list of evenly randomised trials

trials_list = [] 

num_repeats = int(trials_per_block / n_conditions) # how many reps of the 8 conditions you need to fill each block

for block in range(1, n_blocks+1):
    for i in range(num_repeats):
        # randomly shuffle the original 8 conditions
        shuffled_conditions = conditions.sample(frac=1).reset_index(drop=True)
        shuffled_conditions['extra_trial'] = False
        shuffled_conditions['block_nr'] = block
        # append the shuffled chunk to trial list
        trials_list.append(shuffled_conditions)  
    
    # at the end of each block, select, shuffle, and append extra trials needed
    shuffled_extra_trials = conditions.sample(n=extra_trials_per_block, replace=True).reset_index(drop=True)
    shuffled_extra_trials['extra_trial'] = True # mark these as extra trials to avoid scheduling interruption here later
    shuffled_extra_trials['block_nr'] = block
    trials_list.append(shuffled_extra_trials)

# transform trial list into trial df
trials_df = pd.concat(trials_list, ignore_index=True) 

In [136]:
## Generate and evenly shuffle possible interruption lengths
lengths = []

if interrupt_per_block % 2 == 0:
    for block in range(n_blocks): # create new shuffled list per block
        block_lengths = [1] * int(interrupt_per_block/2) + [2] * int(interrupt_per_block/2) # interruption task can be 1 or 2 sums long
        random.shuffle(block_lengths)
        lengths.append(block_lengths) # compile all blocks into one big list of lengths
    print(lengths)
    print(len(lengths)) # should = n_blocks

elif interrupt_per_block % 3 == 0:
    for block in range(n_blocks): # create new shuffled list per block
        block_lengths = [1] * int(interrupt_per_block/3) + [2] * int(interrupt_per_block/3) + [3] * int(interrupt_per_block/3) # interruption task can be 1, 2, or 3 sums long
        random.shuffle(block_lengths)
        lengths.append(block_lengths) # compile all blocks into one big list of lengths
    print(lengths)
    print(len(lengths)) # should = n_blocks

[[1, 2, 1, 2, 2, 1, 1, 1, 2, 2, 1, 2], [2, 2, 1, 2, 1, 1, 2, 1, 1, 1, 2, 2], [1, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2], [2, 2, 1, 1, 2, 1, 2, 1, 2, 1, 2, 1], [2, 2, 1, 2, 1, 1, 1, 1, 2, 2, 1, 2], [1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 2], [1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 2, 2], [1, 2, 1, 1, 1, 2, 2, 1, 2, 2, 2, 1]]
8


In [137]:
## Evenly shuffle possible gap lengths
gaps = []

for block in range(n_blocks):

    # create a list as long as interrupt_per_block that contains equal repeats of each number in gap_range
    block_gaps = gap_range * (interrupt_per_block // len(gap_range))
    print(block_gaps)

    random.shuffle(block_gaps)
    gaps.append(block_gaps)

print(gaps)
print(len(gaps))

[8, 10, 12, 8, 10, 12, 8, 10, 12, 8, 10, 12]
[8, 10, 12, 8, 10, 12, 8, 10, 12, 8, 10, 12]
[8, 10, 12, 8, 10, 12, 8, 10, 12, 8, 10, 12]
[8, 10, 12, 8, 10, 12, 8, 10, 12, 8, 10, 12]
[8, 10, 12, 8, 10, 12, 8, 10, 12, 8, 10, 12]
[8, 10, 12, 8, 10, 12, 8, 10, 12, 8, 10, 12]
[8, 10, 12, 8, 10, 12, 8, 10, 12, 8, 10, 12]
[8, 10, 12, 8, 10, 12, 8, 10, 12, 8, 10, 12]
[[8, 10, 8, 10, 12, 10, 12, 12, 10, 12, 8, 8], [8, 10, 10, 12, 10, 8, 12, 12, 8, 8, 10, 12], [8, 8, 10, 8, 10, 12, 10, 12, 12, 8, 10, 12], [8, 10, 10, 8, 12, 8, 8, 10, 12, 10, 12, 12], [8, 10, 8, 12, 8, 10, 12, 8, 12, 10, 10, 12], [8, 10, 12, 8, 12, 8, 8, 10, 10, 10, 12, 12], [10, 8, 10, 10, 12, 8, 8, 12, 12, 12, 10, 8], [10, 8, 8, 12, 10, 12, 10, 12, 8, 12, 10, 8]]
8


In [138]:
## schedule which trial_n are interruption trials (skipping extra trials at the end of blocks)

interruption_trials = []
interruption_lengths = []

for block in range(n_blocks):   # pick interruption trial idx for each block
    current_trial = gaps[block][-1] + ((trials_per_block + extra_trials_per_block) * block)    # at the start of each block, pick a new trial idx to assign first interruption.
                                    # in gaps, in the list corresponding to the current block nr, pick the last item.
                                    # cannot pick the first item or a random or fixed number because it will mess up the balance of gaps in that list/block, 
                                    # and might cause eventually fewer tot interruptions than planned.
                                    # need to pick last item because it's the only one that isn't used for updating the current_trial counter.
    block_interruption_trials = []
    block_interruption_lengths = []

    for idx_i in range(interrupt_per_block):    # iterate over the nr of interrupt per block
        if current_trial > int(trials_per_block + ((trials_per_block + extra_trials_per_block) * block)):
            break   # stop loop at the end of each block
        block_interruption_trials.append(current_trial)             # store current interruption trial idx
        block_interruption_lengths.append(lengths[block][idx_i])    # store current interruption length
        current_trial += gaps[block][idx_i]                         # update counter by adding current gap to current trial idx
    
    interruption_trials.extend(block_interruption_trials)
    interruption_lengths.extend(block_interruption_lengths)

print(interruption_trials)
print(len(interruption_trials))
print(interruption_lengths)
print(len(interruption_lengths))

[8, 16, 26, 34, 44, 56, 66, 78, 90, 100, 112, 120, 142, 150, 160, 170, 182, 192, 200, 212, 224, 232, 240, 250, 272, 280, 288, 298, 306, 316, 328, 338, 350, 362, 370, 380, 402, 410, 420, 430, 438, 450, 458, 466, 476, 488, 498, 510, 532, 540, 550, 558, 570, 578, 588, 600, 608, 620, 630, 640, 662, 670, 680, 692, 700, 712, 720, 728, 738, 748, 758, 770, 788, 798, 806, 816, 826, 838, 846, 854, 866, 878, 890, 900, 918, 928, 936, 944, 956, 966, 978, 988, 1000, 1008, 1020, 1030]
96
[1, 2, 1, 2, 2, 1, 1, 1, 2, 2, 1, 2, 2, 2, 1, 2, 1, 1, 2, 1, 1, 1, 2, 2, 1, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 1, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 2, 1, 1, 1, 1, 2, 2, 1, 2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 2, 2, 1, 2, 1, 1, 1, 2, 2, 1, 2, 2, 2, 1]
96


In [139]:
## revert gaps and lengths into flat lists instead of lists of lists
gaps = [item for innerList in gaps for item in innerList]
lengths = [item for innerList in lengths for item in innerList]
print(gaps)
print(lengths)

[8, 10, 8, 10, 12, 10, 12, 12, 10, 12, 8, 8, 8, 10, 10, 12, 10, 8, 12, 12, 8, 8, 10, 12, 8, 8, 10, 8, 10, 12, 10, 12, 12, 8, 10, 12, 8, 10, 10, 8, 12, 8, 8, 10, 12, 10, 12, 12, 8, 10, 8, 12, 8, 10, 12, 8, 12, 10, 10, 12, 8, 10, 12, 8, 12, 8, 8, 10, 10, 10, 12, 12, 10, 8, 10, 10, 12, 8, 8, 12, 12, 12, 10, 8, 10, 8, 8, 12, 10, 12, 10, 12, 8, 12, 10, 8]
[1, 2, 1, 2, 2, 1, 1, 1, 2, 2, 1, 2, 2, 2, 1, 2, 1, 1, 2, 1, 1, 1, 2, 2, 1, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 1, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 2, 1, 1, 1, 1, 2, 2, 1, 2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 2, 2, 1, 2, 1, 1, 1, 2, 2, 1, 2, 2, 2, 1]


In [140]:
## generate and evenly shuffle a list of post interrupt target saliency trial types (1 for target salient, else 0)
chunk_length = 6    # can choose largest even factor of tot_interruptions (or of interrupt_per_block if you want to counterbalance within blocks)
trial_saliency_chunks = [0, 1] * int(chunk_length / 2)  # create a chunk with equal reps of 0s and 1s
print(trial_saliency_chunks)

trial_saliency = []

for i in range(int(tot_interruptions/chunk_length)):
        random.shuffle(trial_saliency_chunks)
        trial_saliency.extend(trial_saliency_chunks)

# uncomment below if you want to counterbalance within blocks - nb: this will give less variation in shuffling
'''for block in range(n_blocks):
    block_trial_saliency = []
    for i in range(int(interrupt_per_block/chunk_length)):
        random.shuffle(trial_saliency_chunks)
        block_trial_saliency.extend(trial_saliency_chunks)
    trial_saliency.extend(block_trial_saliency)'''

print(trial_saliency)
print(len(trial_saliency))

[0, 1, 0, 1, 0, 1]
[0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1]
96


In [141]:
## store all lists made above into a tuple
interruption_data = list(zip(interruption_trials, interruption_lengths, trial_saliency))
interruption_data.sort()  # Sort by trial index
print(interruption_data)
print(len(interruption_data))

[(8, 1, 0), (16, 2, 0), (26, 1, 1), (34, 2, 0), (44, 2, 1), (56, 1, 1), (66, 1, 1), (78, 1, 1), (90, 2, 0), (100, 2, 0), (112, 1, 1), (120, 2, 0), (142, 2, 1), (150, 2, 0), (160, 1, 1), (170, 2, 0), (182, 1, 0), (192, 1, 1), (200, 2, 0), (212, 1, 1), (224, 1, 1), (232, 1, 0), (240, 2, 0), (250, 2, 1), (272, 1, 0), (280, 2, 0), (288, 2, 0), (298, 1, 1), (306, 2, 1), (316, 2, 1), (328, 2, 1), (338, 1, 1), (350, 1, 0), (362, 1, 1), (370, 1, 0), (380, 2, 0), (402, 2, 0), (410, 2, 0), (420, 1, 1), (430, 1, 1), (438, 2, 0), (450, 1, 1), (458, 2, 0), (466, 1, 0), (476, 2, 0), (488, 1, 1), (498, 2, 1), (510, 1, 1), (532, 2, 1), (540, 2, 0), (550, 1, 0), (558, 2, 1), (570, 1, 0), (578, 1, 1), (588, 1, 1), (600, 1, 1), (608, 2, 0), (620, 2, 0), (630, 1, 1), (640, 2, 0), (662, 1, 1), (670, 1, 0), (680, 1, 0), (692, 1, 0), (700, 2, 1), (712, 2, 1), (720, 2, 1), (728, 2, 0), (738, 2, 0), (748, 1, 1), (758, 1, 0), (770, 2, 1), (788, 1, 1), (798, 2, 1), (806, 1, 0), (816, 1, 0), (826, 1, 1), (838, 2,

In [142]:
## make full interruption schedule

trial_data = []

since_last_interrupt = None

next_interrupts = [x[0] for x in interruption_data] # list of interrupt trial numbers _ could also simply take interruption_lengths ...?

interrupt_idx = 0 # begin at first interrupt in the list

for trial in range(0, tot_trials + (extra_trials_per_block * n_blocks)):
    
    is_interruption = 0
    interrupt_length = 0
    interrupt_saliency = None
    take_break = False
    
    # check if it's time for a block break every 130 trials
    if trial > 0 and trial % (trials_per_block + extra_trials_per_block) == 0:
        take_break = True

    if take_break == True:
        since_last_interrupt = None # do not update this counter for first trial sequence in each block

    # check if interruption trial matches with current trial 
    if interrupt_idx < len(interruption_data) and interruption_data[interrupt_idx][0] == trial: 
        is_interruption = 1
        interrupt_length = interruption_data[interrupt_idx][1]      # get length
        interrupt_saliency = interruption_data[interrupt_idx][2]    # get trial saliency
        since_last_interrupt = 1                                    # reset timer     
        interrupt_idx += 1                                          # move to next interruption in the list
    else:
        if since_last_interrupt is not None:        # if an interruption has occured before
            since_last_interrupt += 1               # increase counter since last interruption

    # calculate how many trials away the next interruption is (if there are future interruptions)
    future_interrupts = [i for i in next_interrupts if i > trial] # list of upcoming interruptions that happen after current trial

    if future_interrupts:
        until_next_interrupt = trial - future_interrupts[0]
    else:
        until_next_interrupt = None

    trial_data.append({
        'trial_nr': trial,
        'is_interruption_trial': is_interruption,
        'interruption_duration': interrupt_length,
        'post_interrupt_saliency': interrupt_saliency,
        'since_last_interr': since_last_interrupt,
        'until_next_interr': until_next_interrupt,
        'time_to_break' : take_break
    })

df = pd.DataFrame(trial_data)
df.to_csv("onesac_interruptions_list.csv", index=False)

In [143]:
##join the big shuffled conditions dataframe and the interruption dataframe into one dataframe
big_df = pd.concat([trials_df, df], axis=1)
big_df.loc[big_df['extra_trial']==True, 'until_next_interr'] = None # erase this counter for extra trials at end of each block
big_df.to_csv("onesac_concat_cards_bookmarks.csv", index=False)

In [144]:
## check and correct mismatches where target_salient doesn't match with scheduled post-interrupt salience

big_df_swapped = big_df.copy()

mismatches = big_df_swapped[
    (big_df_swapped['is_interruption_trial'] == 1) &
    (big_df_swapped['target_salient'] != big_df_swapped['post_interrupt_saliency'])
]
print(f"Initial mismatches on interruption rows: {len(mismatches)}")

for i in range(len(big_df_swapped)):
    if (
        big_df_swapped.loc[i, 'is_interruption_trial'] == 1 and
        big_df_swapped.loc[i, 'target_salient'] != big_df_swapped.loc[i, 'post_interrupt_saliency']
    ):
        
        # define range of previous n rows (or fewer if i < n); n must be a number < min gap range to avoid interfering with prev sequence
        bracket_start = max(0, i - (min(gap_range) - 1))
        candidates = list(range(bracket_start, i))
        random.shuffle(candidates)  # shuffle in place (so you don't always pick the first or immediately preceding candidate)
        
        # look backward for a match on target_salient
        for j in candidates:
            if big_df_swapped.loc[j, 'target_salient'] == big_df_swapped.loc[i, 'post_interrupt_saliency']:
                # swap values from condition variables columns without swapping any interruption scheduler columns
                for col in big_df_swapped.columns:
                    if (col != 'extra_trial' and
                        col != 'block_nr' and
                        col != 'trial_nr' and
                        col != 'is_interruption_trial' and
                        col != 'interruption_duration' and
                        col != 'post_interrupt_saliency' and
                        col != 'since_last_interr' and
                        col != 'until_next_interr' and
                        col != 'time_to_break'):

                        big_df_swapped.loc[i, col], big_df_swapped.loc[j, col] = big_df_swapped.loc[j, col], big_df_swapped.loc[i, col]

                break  # stop after first match

mismatches = big_df_swapped[
    (big_df_swapped['is_interruption_trial'] == 1) &
    (big_df_swapped['target_salient'] != big_df_swapped['post_interrupt_saliency'])
]
print(f"Remaining mismatches on interruption rows: {len(mismatches)}")

big_df_swapped.to_csv("trial_sequence_left.csv", index=False)

Initial mismatches on interruption rows: 53
Remaining mismatches on interruption rows: 0
